# DEE — Defectos extendidos (Test 2B)

**Motivación:** los defectos puntuales no localizaron modos, lo cual es consistente con el teorema de Lifshitz en 3D: se requiere umbral para ligar estados. Los defectos extendidos (1D, 2D, 3D con fuerza alta) son mejores candidatos.

**Tres configuraciones:**
1. **Dislocación (1D):** línea de nodos a lo largo del eje x con constantes modificadas
2. **Plano defectuoso (2D):** plano z=const con constantes modificadas
3. **Inclusión esférica (3D):** burbuja de nodos con constantes fuertemente modificadas

Comparamos cada uno contra el cristal perfecto y buscamos modos localizados.

---

In [ ]:
import os
import numpy as np
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import time

PLOT_DIR = 'plots_defectos_extendidos'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(name, fig=None, dpi=120):
    if fig is None: fig = plt.gcf()
    path = os.path.join(PLOT_DIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  → {path}')

print('Setup listo')

In [ ]:
# Funciones base
def periodic_distance_matrix(points, L=1.0):
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    return np.linalg.norm(diff, axis=-1)

def grid_perturbed_T3(N_target, jitter_fraction, seed=42):
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    N_actual = n_side**3
    spacing = 1.0 / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter_fraction*spacing, jitter_fraction*spacing, size=points.shape)
    points = points % 1.0
    return points, N_actual

def build_laplacian_with_force_modifiers(points, k_neighbors=30, alpha=2.0,
                                           L=1.0, force_modifiers=None):
    """
    Construye Laplaciano con modificadores de fuerza por nodo.
    force_modifiers: dict {nodo_idx: factor}
    El factor afecta los acoplamientos w_ij → factor · w_ij para enlaces del nodo.
    """
    D_mat = periodic_distance_matrix(points, L=L)
    N = len(points)
    
    if force_modifiers is None:
        force_modifiers = {}
    
    D_search = D_mat.copy()
    np.fill_diagonal(D_search, np.inf)
    neighbor_idx = np.argpartition(D_search, k_neighbors, axis=1)[:, :k_neighbors]
    
    rows, cols, vals = [], [], []
    for i in range(N):
        for j in neighbor_idx[i]:
            d = D_mat[i, j]
            if d > 0:
                w = 1.0 / d**alpha
                # Modificador efectivo: sqrt(f_i · f_j) para simetría
                mod_i = force_modifiers.get(i, 1.0)
                mod_j = force_modifiers.get(j, 1.0)
                w = w * np.sqrt(mod_i * mod_j)
                rows.append(i); cols.append(j); vals.append(w)
    
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    degrees = np.array(W.sum(axis=1)).flatten()
    return diags(degrees) - W

# Construcción del grafo base
N_TARGET = 1331
K_NEIGHBORS = 30
JITTER = 0.1

points, N = grid_perturbed_T3(N_TARGET, JITTER, seed=42)
L_perfect = build_laplacian_with_force_modifiers(points, K_NEIGHBORS, alpha=2.0)

# Referencia: cristal perfecto
t0 = time.time()
eigs_perfect, vecs_perfect = eigsh(L_perfect, k=60, which='SM', tol=1e-8)
idx_s = np.argsort(eigs_perfect)
eigs_perfect = eigs_perfect[idx_s]
vecs_perfect = vecs_perfect[:, idx_s]
omegas_perfect = np.sqrt(eigs_perfect[eigs_perfect > 1e-3])
print(f'Cristal perfecto: {time.time()-t0:.1f}s')
print(f'Primeros 10 ω: {omegas_perfect[:10].round(3)}')

## Función auxiliar: participation ratio

In [ ]:
def participation_ratios(eigenvectors):
    """Calcula participation ratio de cada autovector (más bajo = más localizado)."""
    parts = []
    for i in range(eigenvectors.shape[1]):
        v = eigenvectors[:, i]
        v = v / np.linalg.norm(v)
        v2 = v**2
        ipr = np.sum(v2**2)
        parts.append(1.0 / ipr if ipr > 0 else len(v))
    return np.array(parts)

parts_perfect = participation_ratios(vecs_perfect)
print(f'Participation ratios cristal perfecto, primeros 15:')
print(f'  {parts_perfect[:15].round(1)}')

## Defecto 1 — Dislocación (línea 1D)

Tomamos un tubo cilíndrico estrecho a lo largo del eje x en (y=0.5, z=0.5) con radio 0.08, y modificamos las constantes de fuerza de todos los nodos dentro por factor 5 (duro).

In [ ]:
# Identificar nodos dentro de un cilindro alrededor del eje x central
y_center, z_center = 0.5, 0.5
dislocation_radius = 0.08

# Distancia al eje (y, z) con métrica periódica
dy = points[:, 1] - y_center
dz = points[:, 2] - z_center
dy = dy - np.round(dy)
dz = dz - np.round(dz)
dist_to_axis = np.sqrt(dy**2 + dz**2)

dislocation_nodes = np.where(dist_to_axis < dislocation_radius)[0]
print(f'Dislocación: {len(dislocation_nodes)} nodos en el tubo')
print(f'({len(dislocation_nodes)/N*100:.1f}% del total)')

# Factor fuerte para superar el umbral de Lifshitz
DISLOC_FACTOR = 5.0
force_mods_disloc = {int(i): DISLOC_FACTOR for i in dislocation_nodes}

t0 = time.time()
L_disloc = build_laplacian_with_force_modifiers(points, K_NEIGHBORS, alpha=2.0,
                                                  force_modifiers=force_mods_disloc)
eigs_disloc, vecs_disloc = eigsh(L_disloc, k=60, which='SM', tol=1e-8)
idx_s = np.argsort(eigs_disloc)
eigs_disloc = eigs_disloc[idx_s]
vecs_disloc = vecs_disloc[:, idx_s]
omegas_disloc = np.sqrt(eigs_disloc[eigs_disloc > 1e-3])
parts_disloc = participation_ratios(vecs_disloc)
print(f'Dislocación: {time.time()-t0:.1f}s')

## Defecto 2 — Plano defectuoso (2D)

Plano z=0.5 con espesor 0.06 con constantes ×5.

In [ ]:
z_plane = 0.5
plane_thickness = 0.06

# Distancia al plano z=z_plane (sin periodicidad relevante para z=0.5 en cubo [0,1])
dist_to_plane = np.abs(points[:, 2] - z_plane)
# Periodicidad: si el plano estuviera cerca de z=0 o z=1 habría que considerarla,
# pero z=0.5 está centrado entonces no aplica

plane_nodes = np.where(dist_to_plane < plane_thickness/2)[0]
print(f'Plano: {len(plane_nodes)} nodos')
print(f'({len(plane_nodes)/N*100:.1f}% del total)')

PLANE_FACTOR = 5.0
force_mods_plane = {int(i): PLANE_FACTOR for i in plane_nodes}

t0 = time.time()
L_plane = build_laplacian_with_force_modifiers(points, K_NEIGHBORS, alpha=2.0,
                                                force_modifiers=force_mods_plane)
eigs_plane, vecs_plane = eigsh(L_plane, k=60, which='SM', tol=1e-8)
idx_s = np.argsort(eigs_plane)
eigs_plane = eigs_plane[idx_s]
vecs_plane = vecs_plane[:, idx_s]
omegas_plane = np.sqrt(eigs_plane[eigs_plane > 1e-3])
parts_plane = participation_ratios(vecs_plane)
print(f'Plano: {time.time()-t0:.1f}s')

## Defecto 3 — Inclusión esférica (3D) con fuerza alta

Esfera de radio 0.15 centrada en (0.5, 0.5, 0.5) con factor ×10 (fuerte para superar el umbral de Lifshitz en 3D).

In [ ]:
center = np.array([0.5, 0.5, 0.5])
inclusion_radius = 0.15

diff_center = points - center
diff_center = diff_center - np.round(diff_center)
dist_to_center = np.linalg.norm(diff_center, axis=1)

inclusion_nodes = np.where(dist_to_center < inclusion_radius)[0]
print(f'Inclusión: {len(inclusion_nodes)} nodos')
print(f'({len(inclusion_nodes)/N*100:.1f}% del total)')

INCLUSION_FACTOR = 10.0  # fuerte para 3D
force_mods_incl = {int(i): INCLUSION_FACTOR for i in inclusion_nodes}

t0 = time.time()
L_incl = build_laplacian_with_force_modifiers(points, K_NEIGHBORS, alpha=2.0,
                                               force_modifiers=force_mods_incl)
eigs_incl, vecs_incl = eigsh(L_incl, k=80, which='SM', tol=1e-8)
idx_s = np.argsort(eigs_incl)
eigs_incl = eigs_incl[idx_s]
vecs_incl = vecs_incl[:, idx_s]
omegas_incl = np.sqrt(eigs_incl[eigs_incl > 1e-3])
parts_incl = participation_ratios(vecs_incl)
print(f'Inclusión: {time.time()-t0:.1f}s')

## Comparación espectral y de localización

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Panel 1: espectros superpuestos
ax = axes[0, 0]
n_plot = 40
ax.plot(omegas_perfect[:n_plot], 'o', color='steelblue', markersize=8, alpha=0.7,
        label='Cristal perfecto')
ax.plot(omegas_disloc[:n_plot], '^', color='darkgreen', markersize=8, alpha=0.7,
        label=f'Dislocación 1D (×{DISLOC_FACTOR})')
ax.plot(omegas_plane[:n_plot], 's', color='orange', markersize=8, alpha=0.7,
        label=f'Plano 2D (×{PLANE_FACTOR})')
ax.plot(omegas_incl[:n_plot], 'v', color='crimson', markersize=8, alpha=0.7,
        label=f'Inclusión 3D (×{INCLUSION_FACTOR})')
ax.set_xlabel('Índice del modo')
ax.set_ylabel('ω')
ax.set_title('Espectros comparados: perfecto vs defectos extendidos')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 2: Participation ratio vs ω para cada caso
ax = axes[0, 1]
umbral_localizacion = N / 10
ax.semilogy(omegas_perfect[:n_plot], parts_perfect[:n_plot], 'o', color='steelblue',
            markersize=8, alpha=0.7, label='Perfecto')
ax.semilogy(omegas_disloc[:n_plot], parts_disloc[:n_plot], '^', color='darkgreen',
            markersize=8, alpha=0.7, label='Dislocación')
ax.semilogy(omegas_plane[:n_plot], parts_plane[:n_plot], 's', color='orange',
            markersize=8, alpha=0.7, label='Plano')
ax.semilogy(omegas_incl[:n_plot], parts_incl[:n_plot], 'v', color='crimson',
            markersize=8, alpha=0.7, label='Inclusión')
ax.axhline(umbral_localizacion, color='red', linestyle='--',
           label=f'Umbral N/10 = {int(umbral_localizacion)}')
ax.set_xlabel('ω')
ax.set_ylabel('Participation ratio')
ax.set_title('Participation ratio vs ω — bajo valor = localizado')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which='both')

# Panel 3: zoom en participation ratio bajos
ax = axes[1, 0]
all_parts = np.concatenate([parts_disloc[:n_plot], parts_plane[:n_plot], parts_incl[:n_plot]])
all_omegas = np.concatenate([omegas_disloc[:n_plot], omegas_plane[:n_plot], omegas_incl[:n_plot]])
all_labels = (['Dislocación']*n_plot + ['Plano']*n_plot + ['Inclusión']*n_plot)
colors = ['darkgreen']*n_plot + ['orange']*n_plot + ['crimson']*n_plot
markers = ['^']*n_plot + ['s']*n_plot + ['v']*n_plot

for o, p, c, m, l in zip(all_omegas, all_parts, colors, markers, all_labels):
    ax.scatter(o, p, color=c, marker=m, s=60, alpha=0.7)

ax.axhline(umbral_localizacion, color='red', linestyle='--', label='N/10')
ax.axhline(N/5, color='orange', linestyle=':', alpha=0.5, label='N/5')
ax.set_xlabel('ω')
ax.set_ylabel('Participation ratio')
ax.set_title('Zoom: ¿algún punto cruza N/10?')
ax.set_yscale('log')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which='both')

# Panel 4: diferencias espectrales vs cristal perfecto
ax = axes[1, 1]
diff_disloc = omegas_disloc[:n_plot] - omegas_perfect[:n_plot]
diff_plane = omegas_plane[:n_plot] - omegas_perfect[:n_plot]
diff_incl = omegas_incl[:n_plot] - omegas_perfect[:n_plot]

ax.plot(np.arange(n_plot), diff_disloc, '^-', color='darkgreen', label='Dislocación')
ax.plot(np.arange(n_plot), diff_plane, 's-', color='orange', label='Plano')
ax.plot(np.arange(n_plot), diff_incl, 'v-', color='crimson', label='Inclusión')
ax.axhline(0, color='black', lw=1)
ax.set_xlabel('Índice del modo')
ax.set_ylabel('ω_defecto − ω_perfecto')
ax.set_title('Corrimiento espectral inducido por el defecto')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
save_plot('23_defectos_extendidos_comparacion')
plt.show()

In [ ]:
# Análisis cuantitativo: buscar modos con participation ratio bajo
print('='*70)
print('ANÁLISIS CUANTITATIVO — Defectos extendidos')
print('='*70)
print()
print(f'N = {N}, umbral de localización N/10 = {int(N/10)}')
print(f'Cristal perfecto — min participation ratio: {parts_perfect[:50].min():.1f}')
print()

# Para cada defecto, listamos los modos con menor participation ratio
for name, omegas, parts in [('Dislocación 1D', omegas_disloc, parts_disloc),
                              ('Plano 2D', omegas_plane, parts_plane),
                              ('Inclusión 3D', omegas_incl, parts_incl)]:
    print(f'{name}:')
    # Top 10 modos más localizados (menor participation ratio)
    sort_idx = np.argsort(parts[:50])
    print(f'  Min participation: {parts[:50].min():.1f}')
    print(f'  Modos más localizados (top 5):')
    for k, idx in enumerate(sort_idx[:5]):
        print(f'    ω = {omegas[idx]:.3f}, part = {parts[idx]:.1f}')
    
    # Contar modos bajo umbral
    n_localized = np.sum(parts[:50] < N/10)
    n_semi_localized = np.sum(parts[:50] < N/5)
    print(f'  Modos con part < N/10: {n_localized}')
    print(f'  Modos con part < N/5:  {n_semi_localized}')
    print()

## Visualización 3D del modo más localizado de cada defecto

In [ ]:
fig = plt.figure(figsize=(18, 5))

# Para cada defecto, tomar el modo más localizado
configs = [
    ('Dislocación 1D', vecs_disloc, parts_disloc, omegas_disloc, dislocation_nodes,
     'darkgreen'),
    ('Plano 2D', vecs_plane, parts_plane, omegas_plane, plane_nodes, 'orange'),
    ('Inclusión 3D', vecs_incl, parts_incl, omegas_incl, inclusion_nodes, 'crimson')
]

for plot_idx, (name, vecs, parts, omegas, defect_nodes, color) in enumerate(configs):
    # Tomar el modo más localizado (excluyendo modo cero)
    parts_for_sort = parts.copy()
    # Si hay modo cero, su participation será alto; no lo excluimos
    idx_most_local = int(np.argmin(parts_for_sort[:50]))
    v = vecs[:, idx_most_local]
    v2 = v**2
    
    ax = fig.add_subplot(1, 3, plot_idx+1, projection='3d')
    
    # Scatter con |v|² como color
    log_v2 = np.log10(v2 + 1e-12)
    sc = ax.scatter(points[:, 0], points[:, 1], points[:, 2],
                     c=log_v2, s=15, alpha=0.4, cmap='hot')
    
    # Marcar los nodos del defecto
    ax.scatter(points[defect_nodes, 0], points[defect_nodes, 1],
               points[defect_nodes, 2], c='cyan', s=30, alpha=0.8,
               edgecolors='black', linewidths=0.5, label='Defecto')
    
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('z')
    ax.set_title(f'{name}\nω={omegas[idx_most_local]:.3f}, part={parts[idx_most_local]:.1f}',
                 fontsize=11)
    plt.colorbar(sc, ax=ax, shrink=0.6, label='log|v|²')

plt.tight_layout()
save_plot('24_modos_localizados_3d')
plt.show()

In [ ]:
# Perfil radial: ¿la amplitud decae alejándose del defecto?
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for plot_idx, (name, vecs, parts, omegas, defect_nodes, color) in enumerate(configs):
    ax = axes[plot_idx]
    
    idx_most_local = int(np.argmin(parts[:50]))
    v = vecs[:, idx_most_local]
    v2 = v**2
    
    # Distancia de cada nodo al centro del defecto
    defect_center = points[defect_nodes].mean(axis=0)
    diffs = points - defect_center
    diffs = diffs - np.round(diffs)
    dists_to_defect = np.linalg.norm(diffs, axis=1)
    
    # Para dislocación la distancia relevante es al eje x; para plano al plano z;
    # para inclusión al centro. Usamos distancia al centro del defecto como aproximación.
    # Bineado radial
    r_bins = np.linspace(0, 0.5, 20)
    r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])
    v2_mean_by_r = np.zeros(len(r_centers))
    counts = np.zeros(len(r_centers))
    for k, r in enumerate(r_centers):
        mask = (dists_to_defect >= r_bins[k]) & (dists_to_defect < r_bins[k+1])
        if mask.sum() > 0:
            v2_mean_by_r[k] = v2[mask].mean()
            counts[k] = mask.sum()
    
    ax.semilogy(r_centers[counts > 0], v2_mean_by_r[counts > 0],
                'o-', color=color, markersize=8, lw=2)
    ax.axvline(0, color='black', linestyle=':', alpha=0.5)
    ax.set_xlabel('Distancia al centro del defecto')
    ax.set_ylabel('⟨|v|²⟩ promediado radialmente')
    ax.set_title(f'{name}\nω={omegas[idx_most_local]:.3f}')
    ax.grid(alpha=0.3, which='both')

plt.tight_layout()
save_plot('25_perfiles_radiales')
plt.show()

In [ ]:
# Veredicto
print('='*70)
print('VEREDICTO — Defectos extendidos')
print('='*70)
print()

verdict_by_defect = {}
for name, parts in [('Dislocación', parts_disloc),
                     ('Plano', parts_plane),
                     ('Inclusión', parts_incl)]:
    n_localized = np.sum(parts[:50] < N/10)
    n_semi = np.sum(parts[:50] < N/5)
    min_part = parts[:50].min()
    
    if n_localized >= 1:
        status = 'LOCALIZADOS'
    elif n_semi >= 3:
        status = 'semi-localizados'
    else:
        status = 'no localizan'
    
    verdict_by_defect[name] = status
    print(f'  {name}: {n_localized} modos fuertemente localizados, '
          f'{n_semi} semi-localizados (min part = {min_part:.0f}) → {status}')

print()
print('Comparación con cristal perfecto:')
print(f'  Min participation ratio perfecto: {parts_perfect[:50].min():.1f}')
print(f'  Min participation ratio dislocación: {parts_disloc[:50].min():.1f}')
print(f'  Min participation ratio plano: {parts_plane[:50].min():.1f}')
print(f'  Min participation ratio inclusión: {parts_incl[:50].min():.1f}')
print()

n_localizan = sum(1 for v in verdict_by_defect.values() if v == 'LOCALIZADOS')
if n_localizan >= 2:
    print('✓ LOCALIZACIÓN CONFIRMADA en defectos extendidos')
    print('  El sustrato DEE admite modos localizados cuando los defectos tienen')
    print('  dimensionalidad reducida o fuerza alta, consistente con el teorema de Lifshitz.')
    print('  Los modos localizados son candidatos estructurales para "materia" en DEE.')
elif n_localizan == 1:
    print('~ LOCALIZACIÓN PARCIAL')
    print('  Solo un tipo de defecto localiza modos claramente.')
else:
    print('✗ NO SE DETECTA LOCALIZACIÓN FUERTE')
    print('  Ni siquiera con defectos extendidos aparecen modos con part < N/10.')
    print('  El sustrato DEE parece resistente a localización vibracional.')
    print('  Esto puede indicar: (a) acoplamiento de muy largo alcance sobrepasa defectos,')
    print('                       (b) umbral de Lifshitz aún no alcanzado con factor ×10,')
    print('                       (c) propiedad genuina del sustrato (interesante teóricamente).')

In [ ]:
import shutil
shutil.make_archive('plots_defectos_extendidos', 'zip', PLOT_DIR)
print(f'\nZip creado: plots_defectos_extendidos.zip')

try:
    from google.colab import files
    files.download('plots_defectos_extendidos.zip')
    print('→ Descarga iniciada')
except ImportError:
    pass